In [ ]:
# Vertex Cover — Security Instrumentation Placement (Alias-Based Model)

from optilab.aliases import Model, GRB, quicksum
import numpy as np

# -------------------------------------------------------------------------------
# Model
# -------------------------------------------------------------------------------
m = Model()

# -------------------------------------------------------------------------------
# Problem Data (Enterprise Service Communication Graph)
# -------------------------------------------------------------------------------
rng = np.random.default_rng(47)

n_services = rng.integers(20, 50)

# Random communication graph (EAST-WEST service traffic)
# adjacency matrix: a[i][j] = 1 if service i communicates with j
A = rng.integers(0, 2, size=(n_services, n_services))

# enforce undirected structure + remove self-loops
for i in range(n_services):
    A[i, i] = 0
    for j in range(i + 1, n_services):
        A[j, i] = A[i, j]

# ensure connectivity (avoid trivial isolated edges)
for i in range(n_services):
    if A[i].sum() == 0:
        j = rng.integers(0, n_services)
        if j != i:
            A[i, j] = A[j, i] = 1

# uniform monitoring cost (can later be generalized)
cost = np.ones(n_services)

# -------------------------------------------------------------------------------
# Decision Variables
# -------------------------------------------------------------------------------
x = [m.add_binary_var(name=f"x_{i}") for i in range(n_services)]

# -------------------------------------------------------------------------------
# Objective (minimize monitoring overhead)
# -------------------------------------------------------------------------------
m.set_objective(
    quicksum(cost[i] * x[i] for i in range(n_services)),
    GRB.MINIMIZE
)

# -------------------------------------------------------------------------------
# Constraints (Vertex Cover: every communication edge must be monitored)
# -------------------------------------------------------------------------------
for i in range(n_services):
    for j in range(i + 1, n_services):
        if A[i, j] == 1:
            m.add_constraint(x[i] + x[j] >= 1)

# -------------------------------------------------------------------------------
# Solve
# -------------------------------------------------------------------------------
m.solve()

# -------------------------------------------------------------------------------
# Extract Solution
# -------------------------------------------------------------------------------
solution = np.array([v.x for v in x])
selected = np.where(solution > 0.5)[0]

# compute covered edges
covered_edges = 0
total_edges = 0

for i in range(n_services):
    for j in range(i + 1, n_services):
        if A[i, j] == 1:
            total_edges += 1
            if solution[i] > 0.5 or solution[j] > 0.5:
                covered_edges += 1


print(f"Backend:        {m.backend_name}")
print(f"Services:       {n_services}")
print(f"Edges (approx): {int(A.sum() / 2)}")

print(f"\nSelected monitoring points")
print(f"               (services): {selected}")
print(f"Number of agents deployed: {len(selected)}")
print(f"Edges covered:             {covered_edges} / {total_edges}")

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# -------------------------------------------------------------------------------
# Build graph from adjacency matrix
# -------------------------------------------------------------------------------
G = nx.Graph()

for i in range(n_services):
    G.add_node(i)

for i in range(n_services):
    for j in range(i + 1, n_services):
        if A[i, j] == 1:
            G.add_edge(i, j)

# -------------------------------------------------------------------------------
# Node coloring: selected vs not selected
# -------------------------------------------------------------------------------
node_colors = []
for i in range(n_services):
    if solution[i] > 0.5:
        node_colors.append("red")   # monitoring deployed
    else:
        node_colors.append("lightgray")  # uninstrumented

# -------------------------------------------------------------------------------
# Layout (force-directed = typical for service graphs)
# -------------------------------------------------------------------------------
pos = nx.spring_layout(G, seed=42)

# -------------------------------------------------------------------------------
# Draw graph
# -------------------------------------------------------------------------------
plt.figure(figsize=(8, 6))

nx.draw_networkx_edges(G, pos, alpha=0.4)
nx.draw_networkx_nodes(
    G,
    pos,
    node_color=node_colors,
    node_size=500
)

nx.draw_networkx_labels(G, pos, font_size=10)

plt.title("Service Communication Graph — Vertex Cover (Security Monitoring)")
plt.axis("off")
plt.show()